# Programming Project 8 — Solar Panel Power Output
## Curve Fitting & Interpolation
### ENGR 50 — Engineering Computational Methods and Applications

---

**Name:** ___________________________
**Date:** ___________________________

---

## About This Project

A solar energy company recorded a rooftop panel's power output (Watts) every hour on a clear summer day.
You will analyze this data using **three progressively more powerful tools**:

1. **Polynomial curve fitting** (`np.polyfit`) — fit a smooth mathematical curve to the data
2. **Linear interpolation** (`np.interp`) — estimate power at any time using straight lines between points
3. **Cubic spline interpolation** (`CubicSpline`) — a smoother method that can also compute total energy

By working through each tool in order, you will understand *why* engineers upgrade from simple to more powerful methods.

*Run all cells from top to bottom. Add markdown cells to explain your thinking.*


---
## Part (a) — Setup: Imports and Data  `[1 pt]`

**What we are doing:** Loading the tools we need and storing the solar panel data as NumPy arrays.

- `hour` will be our **x-axis**: the time of day (6 = 6:00 AM, 12 = noon, 18 = 6:00 PM)
- `power` will be our **y-axis**: the panel's power output in Watts at each hour

Everything in this project uses these two arrays, so run this cell first.


In [ ]:
# Part (a): Imports and data setup

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.grid'] = True

# Solar panel hourly measurements — June 21 (summer solstice)
# Power is 0 at sunrise (6 AM) and sunset (6 PM)
hour  = np.array([6,  7,  8,   9,   10,  11,  12,  13,  14,  15,  16,  17,  18])
power = np.array([0, 42, 118, 213, 287, 335, 352, 338, 291, 224, 130,  48,   0])

print('Hours:', hour)
print('Power (W):', power)
print(f'Number of data points: {len(hour)}')
print(f'Peak measured power: {power.max()} W at hour {hour[power.argmax()]}')


---
## Part (b) — Polynomial Curve Fitting  `[3 pts]`

**What we are doing:** Fitting a smooth polynomial curve through the data using `np.polyfit`.

A polynomial is a mathematical formula like:

> P(t) = c₀·t⁴ + c₁·t³ + c₂·t² + c₃·t + c₄   ← degree-4 example

`np.polyfit(x, y, degree)` finds the best coefficients (c₀, c₁, ...) so the curve stays as close as possible to your data points.

**R² (R-squared)** measures fit quality — ranges from 0 to 1, closer to 1 is better:

> R² = 1 - Σ(y - y_fit)² / Σ(y - ȳ)²

You will fit **degree-2** (simpler) and **degree-4** (more flexible) and compare which fits better.


In [ ]:
# Part (b): Fit degree-2 and degree-4 polynomials

# --- Step 1: Fit the polynomials ---
# np.polyfit returns coefficients, highest power first
c2 = np.polyfit(hour, power, 2)   # degree-2 (quadratic — a parabola)
c4 = np.polyfit(hour, power, 4)   # degree-4 (more flexible)

print('Degree-2 coefficients:', c2)
print('Degree-4 coefficients:', c4)

# --- Step 2: Evaluate on a fine grid for smooth curves ---
# linspace gives 400 evenly-spaced points — smooth enough to look like a curve
t_fine = np.linspace(5.5, 18.5, 400)
P2_fine = np.polyval(c2, t_fine)
P4_fine = np.polyval(c4, t_fine)

# --- Step 3: Compute R² for each fit ---
# First, evaluate at the original data points (not the fine grid)
def r_squared(y, y_fit):
    """Compute R² — how well does y_fit match the measured y?"""
    ss_res = np.sum((y - y_fit) ** 2)   # sum of squared residuals
    ss_tot = np.sum((y - y.mean()) ** 2) # total variance in the data
    return 1 - ss_res / ss_tot

r2_deg2 = r_squared(power, np.polyval(c2, hour))
r2_deg4 = r_squared(power, np.polyval(c4, hour))

print(f'\nDegree-2 R² = {r2_deg2:.4f}')
print(f'Degree-4 R² = {r2_deg4:.4f}')
print('(Expected: degree-2 ≈ 0.98, degree-4 ≈ 0.9993)')

# --- Step 4: Plot both fits with the original data ---
plt.figure()
plt.plot(hour, power, 'ko', markersize=9, zorder=5, label='Measured data')
plt.plot(t_fine, P2_fine, 'b--', linewidth=2, label=f'Degree-2 fit  (R²={r2_deg2:.4f})')
plt.plot(t_fine, P4_fine, 'r-',  linewidth=2, label=f'Degree-4 fit  (R²={r2_deg4:.4f})')
plt.xlabel('Hour of Day')
plt.ylabel('Power Output (W)')
plt.title('Solar Panel Output: Polynomial Curve Fits')
plt.legend()
plt.tight_layout()
plt.show()


**Part (b) Reflection** — Answer both questions in this cell:

1. Which polynomial degree fits the data better? What does the R² value tell you?
2. Looking at the plot, where does the degree-2 fit disagree most with the data?

> *Your answer here:*


---
## Part (c) — Find Peak Power and Time  `[1 pt]`

**What we are doing:** Using the degree-4 polynomial to find the *exact* peak power output and when it occurs.

We can't just look at the data table — the peak might fall between two measurement hours.
Instead, we evaluate the polynomial on a very fine time grid (1000 points) and use `np.argmax` to find the maximum.

> `np.argmax(array)` → returns the **index** of the largest value in the array.
> Use that index to look up the corresponding hour and power.


In [ ]:
# Part (c): Find peak power using the degree-4 fit on a fine grid

# Create a fine grid from hour 6 to 18 — 1000 points gives good precision
t_fine = np.linspace(6, 18, 1000)
P_fine = np.polyval(c4, t_fine)   # evaluate degree-4 polynomial

# np.argmax returns the index of the largest value
peak_idx  = np.argmax(P_fine)
peak_hour = t_fine[peak_idx]      # look up the hour at that index
peak_pow  = P_fine[peak_idx]      # look up the power at that index

# Convert decimal hour to hours:minutes for display
hours_part   = int(peak_hour)
minutes_part = int((peak_hour % 1) * 60)

print(f'Peak power: {peak_pow:.1f} W')
print(f'Time of peak: {peak_hour:.3f} → {hours_part}:{minutes_part:02d}')
print('(Expected: peak ≈ 354 W near 12:00–12:20)')


---
## Part (d) — Simple Interpolation with `np.interp`  `[2 pts]`

**What we are doing:** Estimating power at three specific times that were *not* directly measured, using the simplest possible method — linear interpolation.

**How np.interp works:**
It finds the two nearest measured hours on either side of your target time, then draws a straight line between them and reads off the value.

> `np.interp(x_new, x_known, y_known)`
> `x_new` can be a single number or an array — it estimates all of them at once.

This is very fast and easy, but notice: it draws **straight lines** between each pair of data points.
In Part (e) you will see how a cubic spline gives a smoother result.

**Times to estimate:**
- 9:30 AM → decimal hour: 9 + 30/60 = **9.5**
- 1:15 PM → decimal hour: 13 + 15/60 = **13.25**
- 4:45 PM → decimal hour: 16 + 45/60 = **16.75**


In [ ]:
# Part (d): Linear interpolation with np.interp

# Define the three query times as decimal hours
t_query = np.array([9 + 30/60,    # 9:30 AM
                    13 + 15/60,   # 1:15 PM
                    16 + 45/60])  # 4:45 PM

labels = ['9:30 AM', '1:15 PM', '4:45 PM']

# np.interp estimates power at each query time using straight-line interpolation
P_linear = np.interp(t_query, hour, power)

# Print a table of results
print('Linear Interpolation Results (np.interp)')
print(f"{'Time':<12} {'Power (W)':>10}")
print('-' * 24)
for lbl, P in zip(labels, P_linear):
    print(f'{lbl:<12} {P:>10.1f}')

# --- Plot: show the straight-line nature of np.interp ---
plt.figure()
plt.plot(hour, power, 'ko-', markersize=9, linewidth=1.5, zorder=5, label='Measured data (linear connect)')
plt.plot(t_query, P_linear, 'b^', markersize=13, zorder=6, label='np.interp estimates')

# Annotate each interpolated point
for lbl, t, P in zip(labels, t_query, P_linear):
    plt.annotate(f'{lbl}\n{P:.0f} W', xy=(t, P),
                 xytext=(t + 0.3, P + 15), fontsize=9,
                 arrowprops=dict(arrowstyle='->', color='blue'))

plt.xlabel('Hour of Day')
plt.ylabel('Power Output (W)')
plt.title('Solar Panel Output: Linear Interpolation (np.interp)')
plt.legend()
plt.tight_layout()
plt.show()


---
## Part (e) — Upgrade to Cubic Spline  `[3 pts]`

**What we are doing:** Replacing the straight-line interpolation with a *cubic spline* — a smooth curve that passes exactly through every data point.

**Why upgrade?**

| Feature | np.interp | CubicSpline |
|---|---|---|
| Passes through data points | ✓ | ✓ |
| Smooth curve (no corners) | ✗ — straight lines | ✓ — smooth arcs |
| Can integrate (area under curve) | ✗ | ✓ |
| Physically realistic for this problem | Less so | Yes |

The integral of the power curve gives **total energy generated** (power × time = energy):

> `cs.integrate(6, 18)` → area under the curve from 6 AM to 6 PM → Watt-hours of energy

---

### Part (e-1): Compare spline vs. linear interpolation


In [ ]:
# Part (e-1): Build the CubicSpline and compare to np.interp

# Build the cubic spline — it passes through all 13 data points
cs = CubicSpline(hour, power)

# Estimate power at the same three times as Part (d)
t_query = np.array([9 + 30/60, 13 + 15/60, 16 + 45/60])
labels  = ['9:30 AM', '1:15 PM', '4:45 PM']
P_linear = np.interp(t_query, hour, power)   # from Part (d)
P_spline = cs(t_query)                        # from CubicSpline

# Print a comparison table
print('Comparison: np.interp vs. CubicSpline')
print(f"{'Time':<12} {'np.interp (W)':>14} {'CubicSpline (W)':>16} {'Difference':>11}")
print('-' * 56)
for lbl, Pl, Ps in zip(labels, P_linear, P_spline):
    print(f'{lbl:<12} {Pl:>14.1f} {Ps:>16.1f} {Ps - Pl:>+11.1f}')

# Plot: both methods on one figure
t_fine = np.linspace(6, 18, 500)

plt.figure()
plt.plot(hour, power, 'ko', markersize=9, zorder=5, label='Measured data')
plt.plot(hour, power, 'k-', linewidth=1.2, alpha=0.4, label='Linear (np.interp)')  # straight lines
plt.plot(t_fine, cs(t_fine), 'g-', linewidth=2.2, label='Cubic spline')
plt.plot(t_query, P_linear, 'b^', markersize=12, zorder=6, label='np.interp estimates')
plt.plot(t_query, P_spline, 'r*', markersize=14, zorder=6, label='CubicSpline estimates')
plt.xlabel('Hour of Day')
plt.ylabel('Power Output (W)')
plt.title('Solar Panel Output: np.interp vs. CubicSpline')
plt.legend()
plt.tight_layout()
plt.show()


**Part (e-1) Reflection** — Answer in this cell:

1. Are the differences between `np.interp` and `CubicSpline` large or small for this dataset?
2. Looking at the plot, where does the difference between straight-line and spline interpolation show up most clearly?
3. Which method do you trust more for this problem, and why?

> *Your answer here:*


---
### Part (e-2): Compute Total Daily Energy


In [ ]:
# Part (e-2): Integrate the cubic spline to find total energy generated

# cs.integrate(a, b) computes the area under the spline curve from hour a to hour b.
# Area under power (W) vs. time (h) curve = energy in Watt-hours (Wh).
E_wh  = cs.integrate(6, 18)    # Watt-hours
E_kwh = E_wh / 1000            # convert to kilowatt-hours
rate  = 0.22                   # electricity cost in $/kWh
daily_savings = E_kwh * rate

print('Total Energy Generated:')
print(f'  {E_wh:.1f} Wh  =  {E_kwh:.3f} kWh')
print()
print(f'Daily Electricity Savings @ ${rate}/kWh:')
print(f'  ${daily_savings:.2f}')
print()
print('(Expected: ~2.2–2.4 kWh, ~$0.48–$0.53)')


---
## Summary Table

Fill in your results below:

| Part | Task | Your Result |
|------|------|-------------|
| (a) | Data loaded successfully | ✓ |
| (b) | Degree-2 R² | |
| (b) | Degree-4 R² | |
| (b) | Which degree fits better? | |
| (c) | Peak power (W) | |
| (c) | Time of peak | |
| (d) | Power at 9:30 AM — np.interp (W) | |
| (d) | Power at 1:15 PM — np.interp (W) | |
| (d) | Power at 4:45 PM — np.interp (W) | |
| (e-1) | Power at 9:30 AM — CubicSpline (W) | |
| (e-1) | Power at 1:15 PM — CubicSpline (W) | |
| (e-1) | Power at 4:45 PM — CubicSpline (W) | |
| (e-2) | Total energy generated (kWh) | |
| (e-2) | Daily savings ($) | |

---
*End of Programming Project 7*
